# 02 — Camada Silver: TransformaçãoLimpeza, padronização, tipagem e construção das dimensões conformadas.**Transformações aplicadas e justificativa**| Transformação | Por quê | Impacto ||---|---|---|| Normalização de CNPJ | Cadastro e informe usam formatos diferentes (com e sem máscara) | Sem isso o join falharia silenciosamente, retornando nulo para todas as classes || Tipagem explícita | Valores para double, competência para date, cotistas para int | Permite agregações e ordenação temporal corretas || Deduplicação pela chave natural | A base traz registros repetidos para a mesma classe e data | Remove duplicatas sem descartar dados legítimos || Filtro de validade | Patrimônio e cota maiores que zero | Remove classes em encerramento que distorceriam médias e retornos || Coluna derivada `captacao_liquida` | Usada em todas as análises de fluxo | Evita recalcular e padroniza o sinal || Particionamento por data | Todas as consultas filtram ou agrupam por período | Reduz leitura de arquivos |

In [ ]:
from pyspark.sql import functions as Ffrom pyspark.sql import Window

## 2.1 Tabela fato**Definição do grão.** O informe diário não é por fundo, mas por classe e subclasse. A chaveé `cnpj_classe` + `id_subclasse` + `data`.Verificou-se empiricamente que, quando uma classe possui subclasses, apenas as linhas desubclasse são publicadas — não existe linha consolidada da classe na mesma data. A ausênciade sobreposição é testada na célula seguinte, antes de qualquer agregação.

In [ ]:
# Teste de sobreposição classe x subclasse: precisa dar zerob = spark.table("workspace.bronze.inf_diario_fi")uma_data = b.filter("DT_COMPTC = (select max(DT_COMPTC) from workspace.bronze.inf_diario_fi)")com_sub = uma_data.filter("ID_SUBCLASSE is not null and ID_SUBCLASSE != ''") \                  .select("CNPJ_FUNDO_CLASSE").distinct()sem_sub = uma_data.filter("ID_SUBCLASSE is null or ID_SUBCLASSE = ''") \                  .select("CNPJ_FUNDO_CLASSE").distinct()print("CNPJs com linha de classe E de subclasse na mesma data:",      com_sub.join(sem_sub, "CNPJ_FUNDO_CLASSE").count())print("(zero confirma que somar o patrimônio não gera dupla contagem dentro do arquivo)")

In [ ]:
fato = (spark.table("workspace.bronze.inf_diario_fi")    .select(        F.regexp_replace("CNPJ_FUNDO_CLASSE", r"[^0-9]", "").alias("cnpj_classe"),        F.coalesce(F.col("ID_SUBCLASSE"), F.lit("")).alias("id_subclasse"),        F.to_date("DT_COMPTC").alias("data"),        F.col("VL_TOTAL").cast("double").alias("valor_carteira"),        F.col("VL_PATRIM_LIQ").cast("double").alias("patrimonio_liquido"),        F.col("VL_QUOTA").cast("double").alias("valor_cota"),        F.col("CAPTC_DIA").cast("double").alias("captacao"),        F.col("RESG_DIA").cast("double").alias("resgate"),        F.col("NR_COTST").cast("int").alias("cotistas"))    .dropDuplicates(["cnpj_classe", "id_subclasse", "data"])    .filter("patrimonio_liquido > 0 and valor_cota > 0")    .withColumn("captacao_liquida", F.col("captacao") - F.col("resgate")))(fato.write.format("delta").mode("overwrite")     .option("overwriteSchema", True)     .partitionBy("data")     .saveAsTable("workspace.silver.fato_informe"))bronze_n = spark.table("workspace.bronze.inf_diario_fi").count()silver_n = fato.count()print("bronze:", bronze_n, "| silver:", silver_n,      "| removidas:", bronze_n - silver_n,      f"({100*(bronze_n-silver_n)/bronze_n:.2f}%)")

## 2.2 Dimensão de classe**Identificação de crédito privado.** Não existe campo booleano na base. Em vez de inferirpelo nome do fundo, usa-se a `Classificacao_Anbima` do registro de classes — critério oficiale auditável.**Identificação de classes feeder.** Classes de investimento em cotas aplicam em outrasclasses, o que produz dupla contagem quando se soma o patrimônio da indústria. A marcação éfeita por padrão na denominação social ("EM COTAS", "FIC"), o que é uma **aproximação**:captura a maioria dos casos, mas não substitui a hierarquia real de investimento, que a basenão disponibiliza.

In [ ]:
CP = ["Renda Fixa Duração Livre Crédito Livre",      "Renda Fixa Duração Livre Grau de Invest.",      "Renda Fixa Duração Baixa Grau de Invest."]classe = (spark.table("workspace.bronze.cad_classe")    .select(        F.regexp_replace("CNPJ_Classe", r"[^0-9]", "").alias("cnpj_classe"),        F.col("ID_Registro_Fundo").alias("id_fundo"),        F.col("Denominacao_Social").alias("nome_classe"),        F.col("Classificacao_Anbima").alias("classificacao_anbima"),        F.col("Classificacao").alias("classificacao_cvm"),        F.col("Situacao").alias("situacao"),        F.to_date("Data_Inicio").alias("data_inicio"))    .withColumn("credito_privado", F.col("classificacao_anbima").isin(CP))    .withColumn("nome_upper", F.upper(F.col("nome_classe")))    .withColumn("classe_feeder",        F.col("nome_upper").contains("EM COTAS") | F.col("nome_upper").rlike(r"\bFIC\b"))    .drop("nome_upper"))fundo = (spark.table("workspace.bronze.cad_fundo")    .select(        F.col("ID_Registro_Fundo").alias("id_fundo"),        F.col("Denominacao_Social").alias("nome_fundo"),        F.col("Administrador").alias("administrador"),        F.col("Gestor").alias("gestor"),        F.col("Tipo_Fundo").alias("tipo_fundo")))dim_classe = (classe.join(fundo, "id_fundo", "left")    .withColumn("id_gestor", F.md5(F.upper(F.trim(F.coalesce("gestor", F.lit("NAO INFORMADO"))))))    .dropDuplicates(["cnpj_classe"]))(dim_classe.write.format("delta").mode("overwrite")    .option("overwriteSchema", True)    .saveAsTable("workspace.silver.dim_classe"))print(dim_classe.count(), "classes")print(dim_classe.filter("credito_privado").count(), "de credito privado")print(dim_classe.filter("credito_privado and classe_feeder").count(), "feeder no recorte")

## 2.3 Dimensão de gestorDimensão separada porque gestor é atributo de análise por si só: permite agregar captação eretorno por casa gestora sem repetir o texto em cada linha da fato. A chave substituta é umhash do nome normalizado, já que a base não fornece identificador de gestor.

In [ ]:
dim_gestor = (spark.table("workspace.silver.dim_classe")    .groupBy("id_gestor")    .agg(F.first("gestor", ignorenulls=True).alias("nome_gestor"),         F.countDistinct("cnpj_classe").alias("n_classes"),         F.countDistinct(F.when(F.col("credito_privado"), F.col("cnpj_classe"))).alias("n_classes_cp"),         F.countDistinct("administrador").alias("n_administradores")))(dim_gestor.write.format("delta").mode("overwrite")    .option("overwriteSchema", True)    .saveAsTable("workspace.silver.dim_gestor"))print(dim_gestor.count(), "gestores")display(dim_gestor.orderBy(F.desc("n_classes_cp")).limit(10))

## 2.4 Dimensão de tempoConstruída a partir das datas efetivamente presentes na fato, não de um calendário sintético —assim cada linha corresponde a um dia com informe publicado.Os atributos de fim de mês e fim de trimestre existem para a Pergunta 5: permitem testar oefeito de calendário sobre resgates de forma objetiva, em vez de por inspeção visual.

In [ ]:
datas = spark.table("workspace.silver.fato_informe").select("data").distinct()w_mes = Window.partitionBy(F.year("data"), F.month("data"))dim_tempo = (datas    .withColumn("ano", F.year("data"))    .withColumn("mes", F.month("data"))    .withColumn("trimestre", F.quarter("data"))    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))    .withColumn("dia_semana", F.date_format("data", "EEEE"))    .withColumn("ultimo_dia_util_mes", F.col("data") == F.max("data").over(w_mes))    .withColumn("primeiro_dia_util_mes", F.col("data") == F.min("data").over(w_mes))    .withColumn("ultimo_dia_util_trimestre",        F.col("ultimo_dia_util_mes") & F.col("mes").isin(3, 6, 9, 12))    .withColumn("dezembro", F.col("mes") == 12))(dim_tempo.write.format("delta").mode("overwrite")    .option("overwriteSchema", True)    .saveAsTable("workspace.silver.dim_tempo"))print(dim_tempo.count(), "dias uteis com informe")display(dim_tempo.orderBy("data").limit(10))